In [1]:
# Package installs go here 
%conda install polars, plotly

/opt/anaconda3/lib/python3.12/site-packages/conda/base/context.py:201: FutureWarning: Adding 'defaults' to channel list implicitly is deprecated and will be removed in 25.3. 

To remove this warning, please choose a default channel explicitly with conda's regular configuration system, e.g. by adding 'defaults' to the list of channels:

  conda config --add channels defaults

For more information see https://docs.conda.io/projects/conda/en/stable/user-guide/configuration/use-condarc.html

  deprecated.topic(
/opt/anaconda3/lib/python3.12/site-packages/conda/base/context.py:201: FutureWarning: Adding 'defaults' to channel list implicitly is deprecated and will be removed in 25.3. 

To remove this warning, please choose a default channel explicitly with conda's regular configuration system, e.g. by adding 'defaults' to the list of channels:

  conda config --add channels defaults

For more information see https://docs.conda.io/projects/conda/en/stable/user-guide/configuration/use-condarc.

In [2]:
# python imports 
import polars as pl
import plotly.io as pio
pio.renderers.default = "notebook_connected"

In [35]:
# Read eBird csv data
ebird = pl.read_csv("data/ebird.csv") 
observations = pl.read_csv("data/observation.csv") 
species = pl.read_csv("data/899_Species.csv") 
header = pl.read_csv(
    "data/header.csv",
    null_values=["NA", ""],
    infer_schema_length=100_000,
    schema_overrides={"k_Latitude": pl.Float64, "k_Longitude": pl.Float64},
)
locations = pl.read_csv(
    "data/location.csv",
    null_values=["NA", ""],
    infer_schema_length=100_000,
    schema_overrides={"k_Latitude": pl.Float64, "k_Longitude": pl.Float64},
)

traits = pl.read_csv(
    "data/traits.csv",
    null_values=["NA", ""],
    infer_schema_length=100_000,
)
synonyms = pl.read_csv("data/synonyms.csv")


In [36]:
%%capture
# Rename variable names in eBird 
# In polars, we can apply a dictionary to rename columns
ebird = ebird.rename({"K-Latitude" : "key_latitude", "K-Longitude": "key_longitude"})
ebird = ebird.rename({"Latitude" : "latitude", "Longitude": "longitude"})

In [37]:
# Helper function that converts form degree:minutes to degree:decimal
def ddmm_to_dd(column_name: str) -> pl.Expr:
    degree = pl.col(column_name) // 100
    minutes = pl.col(column_name) - degree * 100
    return degree + (minutes / 60)

ebird = ebird.with_columns(
    key_latitude = -ddmm_to_dd("key_latitude"), 
    key_longitude = ddmm_to_dd("key_longitude")
)

ebird = ebird.with_columns(
    latitude = -ddmm_to_dd("latitude"), 
    longitude = ddmm_to_dd("longitude")
)




In [38]:
# Change ebird start and end dates from string to date type (format is YYYY/MM/DD)
ebird = ebird.with_columns(
    pl.col("Observation Start Date", "Observation End Date").str.to_date("%Y/%m/%d")
)

## Slide 1 - Sydney Heatmap
The following graph is a heatmap of bird occurences in sydney...

In [8]:
import plotly.figure_factory as ff

# Approximate Greater Sydney extent (Blue Mountains to coast, Central Coast to Wollondilly)
SYD_LAT = (-34.35, -32.99)
SYD_LON = (149.97, 151.63)

sydney_ebird = ebird.filter(
    pl.col("key_latitude").is_between(*SYD_LAT),
    pl.col("key_longitude").is_between(*SYD_LON),
)

fig = ff.create_hexbin_mapbox(
    data_frame=sydney_ebird, lat="key_latitude", lon="key_longitude",
    nx_hexagon=10, opacity=0.5, labels={"color": "Point Count"},
    mapbox_style="open-street-map"
)

fig.update_layout(margin = dict(b = 0, t = 0, l = 0, r = 0)) 
fig.show()


The hotspots are wollemi national park, hornsby and wollogong.

## eBird Observation Histogram

In [ ]:
# Code for observation histogram


datetime.timedelta(0)

- The *earliest* observation: 27th August 1951 
- The *latest* observation start is 31st March 2020 

- Dates are strings

## eBird heatmap

In [ ]:
# Code for eBird Heatmap

## DES Heatmap

In [ ]:
# Code for DES Heatmap